# Results Analysis

CPU-only ноутбук для проверки уже сохраненных результатов генерации и оценки. Его удобно запускать после GPU-прогонов, чтобы быстро посмотреть CSV-файлы, пересчитать стилометрию и подготовить шаблон ручной оценки.

## 1. Подготовка репозитория

Эту ячейку нужно выполнить первой. В чистом Colab runtime она скачает публичный GitHub-репозиторий и перейдет в корень проекта. GPU для этого ноутбука не нужен.

In [ ]:
# Подготовка окружения Colab.
# Если ноутбук открыт не из корня репозитория, эта ячейка скачает публичный GitHub-репозиторий
# и переключит рабочую директорию на /content/llama_test.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/chipalgash/llama_test.git"
REPO_DIR = Path("/content/llama_test")

if not Path("src").exists():
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} -> {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Already in project root; using current working directory.")

print("Current directory:", Path.cwd())
assert Path("src").exists(), "src/ not found. Check repository clone or %cd to project root."
assert Path("data/processed/en/eval_payload.jsonl").exists(), "EN eval payload is missing."
assert Path("data/processed/ru/eval_payload.jsonl").exists(), "RU eval payload is missing."
print("Repository is ready.")

In [ ]:
# Быстрая проверка наличия файлов генерации.
# Если CSV уже есть в outputs/, показываем первые строки для визуального контроля.
import pandas as pd
from pathlib import Path

for path in [
    Path('outputs/en/baseline_generations.csv'),
    Path('outputs/en/finetuned_generations.csv'),
    Path('outputs/ru/baseline_generations.csv'),
    Path('outputs/ru/finetuned_generations.csv'),
]:
    print(path, 'exists=' + str(path.exists()))
    if path.exists():
        display(pd.read_csv(path).head())

## 2. Пересчет стилометрии после генерации

Запускайте эту ячейку, когда CSV с baseline и fine-tuned генерациями уже лежат в `outputs/en/` и `outputs/ru/`. Она обновит таблицы признаков и сводные метрики для обеих языковых веток.

In [ ]:
# Пересчет автоматических метрик для английского и русского прогонов.
# Если какой-то CSV еще не создан, соответствующая команда завершится ошибкой с указанием отсутствующего файла.
!PYTHONPATH=$PWD python -m src.evaluation.run_stylometry \
  --language en \
  --human-jsonl data/processed/en/eval_payload.jsonl \
  --baseline-csv outputs/en/baseline_generations.csv \
  --finetuned-csv outputs/en/finetuned_generations.csv \
  --features-csv outputs/en/stylometric_features.csv \
  --summary-csv outputs/en/metric_summary.csv

!PYTHONPATH=$PWD python -m src.evaluation.run_stylometry \
  --language ru \
  --human-jsonl data/processed/ru/eval_payload.jsonl \
  --baseline-csv outputs/ru/baseline_generations.csv \
  --finetuned-csv outputs/ru/finetuned_generations.csv \
  --features-csv outputs/ru/stylometric_features.csv \
  --summary-csv outputs/ru/metric_summary.csv

## 3. Шаблон ручной оценки

Эта ячейка создает CSV-шаблон для экспертной/ручной оценки текстов. Его можно использовать для сравнения human, baseline и fine-tuned образцов по качественным критериям.

In [ ]:
# Создание шаблона ручной оценки.
# Скрипт берет сгенерированные тексты и формирует таблицу, которую можно отдать разметчику.
!PYTHONPATH=$PWD python -m src.evaluation.manual_eval_template